[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1BoazozMT52he6rRSPq14zbG8UzwFG_cj/view?usp=drive_link)

# Workflow Evaluation

This notebook evaluates multi-agent workflows running on the FloTorch gateway using a DAG JSON file and a dataset JSON file.

**FloTorch Console:** [https://docs.flotorch.cloud/introduction/](https://docs.flotorch.cloud/introduction/)

**Prerequisites**
- Deploy workflow agents in FloTorch and use their deployed names in the DAG JSON.
- Create a FloTorch API key in Workspace Settings.
- Install Floeval with FloTorch support.

**Objectives**
- Load workflow DAG configuration from a JSON file
- Load partial workflow dataset from a JSON file
- Run the workflow on all samples and build full traced samples
- Evaluate workflow outputs with agent metrics

## 1. Installation

Install Floeval with FloTorch support. Required for evaluating multi-agent workflows on the FloTorch gateway.

In [ ]:
%pip install floeval[flotorch]>=0.2.0b1

## 2. Configuration Constants

Set gateway credentials and model IDs from your FloTorch workspace.

In [ ]:
import getpass

# FloTorch gateway configuration
FLOTORCH_BASE_URL = "https://gateway.flotorch.cloud/openai/v1"
FLOTORCH_API_KEY = getpass.getpass("Enter your FloTorch API key: ")
FLOTORCH_CHAT_MODEL = "<flotorch-chat-model>"
FLOTORCH_EMBEDDING_MODEL = "<flotorch-embedding-model>"

## 3. Imports

Import agent evaluation components, dataset schemas, `WorkflowRunner`, and LLM configuration schema.

In [ ]:
import json
from pathlib import Path

from floeval.api.agent_evaluation import AgentEvaluation
from floeval.api.dataset_loaders.agent_file_loader import AgentDatasetLoader
from floeval.config.schemas.io.agent_dataset import AgentDataset
from floeval.config.schemas.io.llm import OpenAIProviderConfig
from floeval.flotorch import WorkflowRunner

## 4. Configure the LLM

The LLM configuration is built for the FloTorch gateway. Set `FLOTORCH_BASE_URL` and `FLOTORCH_API_KEY` in your environment or pass them explicitly. Credentials are available from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=FLOTORCH_BASE_URL,
    api_key=FLOTORCH_API_KEY,
    chat_model=FLOTORCH_CHAT_MODEL,
    embedding_model=FLOTORCH_EMBEDDING_MODEL,
)

## 5. Load Workflow DAG Config (JSON)

Define workflow structure in a DAG JSON file.

Minimal JSON shape:

```json
{
  "uid": "sequential-workflow-001",
  "name": "Sequential Workflow",
  "nodes": [
    {"id": "start", "type": "START", "label": "Start"},
    {"id": "agent1", "type": "AGENT", "label": "Agent 1", "callableName": "agent1:latest"},
    {"id": "end", "type": "END", "label": "End"}
  ],
  "edges": [
    {"sourceNodeId": "start", "targetNodeId": "agent1"},
    {"sourceNodeId": "agent1", "targetNodeId": "end"}
  ]
}
```

**Example file**  
<a href="../datasets/agentic_workflow/sample_workflow_dag.json" download="sample_workflow_dag.json">sample_workflow_dag.json</a>

In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your workflow DAG JSON file:")
    uploaded_dag = files.upload()
    if not uploaded_dag:
        raise RuntimeError("No DAG file uploaded.")
    dag_path = Path(next(iter(uploaded_dag.keys())))
else:
    dag_path = Path(input("Enter path to workflow DAG JSON file: ").strip().strip('"')).expanduser()

with dag_path.open("r", encoding="utf-8") as f:
    dag_config = json.load(f)

print(f"DAG config loaded from {dag_path}")

## 6. Create the Workflow Runner

The `WorkflowRunner` is instantiated with the DAG config and LLM config. It executes the workflow by calling each agent node according to the DAG edges.

In [ ]:
runner = WorkflowRunner(dag_config=dag_config, llm_config=llm_config)
print("WorkflowRunner created")

## 7. Load Workflow Dataset (JSON)

Minimal JSON shape (partial workflow samples):

```json
{
  "samples": [
    { "user_input": "...", "reference_outcome": "..." }
  ]
}
```

**Example file**  
<a href="../datasets/agentic_workflow/sample_workflow_partial_dataset.json" download="sample_workflow_partial_dataset.json">sample_workflow_partial_dataset.json</a>

### Resolve Workflow Dataset Path

This step accepts a dataset file upload in Colab or a local JSON path in Jupyter.

In [ ]:
try:
    from google.colab import files
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your agent dataset JSON file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = Path(input("Enter path to agent dataset JSON file: ").strip().strip('"')).expanduser()


### Load Workflow Dataset

Load workflow test cases with `AgentDatasetLoader.from_file(...)`.

In [ ]:
dataset = AgentDatasetLoader.from_file(dataset_path)
print(f"Dataset loaded from {dataset_path}: {len(dataset.samples)} sample(s)")


## 8. Run Workflow to Produce Full Samples

Execute the workflow on partial samples to generate full samples with traces.

### Execute Workflow on Dataset (Async)

Run `runner.run_on_dataset(dataset.all_partial)` and wrap the returned full samples in `AgentDataset` for scoring.


In [ ]:
full_samples = await runner.run_on_dataset(dataset.all_partial)
full_dataset = AgentDataset(samples=full_samples)


### Build Agent Evaluation

Configure `AgentEvaluation` with the full traced dataset and workflow-level metrics.


In [ ]:
evaluation = AgentEvaluation(
    dataset=full_dataset,
    llm_config=llm_config,
    metrics=["goal_achievement", "response_coherence", "ragas:agent_goal_accuracy"],
    default_provider="builtin",
)


### Run Evaluation

Compute workflow metric scores and store them in `results`.


In [ ]:
results = evaluation.run()


### Attach Workflow Trace Metadata

Add `agent_traces` and `workflow_execution` metadata to each `sample_results` row, then print summary metrics.


In [ ]:
# Enrich sample_results with agent_traces from full samples (per user guide)
for i, row in enumerate(results.sample_results):
    if i < len(full_samples):
        s = full_samples[i]
        row["agent_traces"] = s.agent_traces or []
        row["workflow_execution"] = (s.metadata or {}).get("workflow_execution")

print("Summary:", results.summary)


## 9. Inspect Per-Sample Results

Each sample result includes `final_response` and metric scores. This enables per-sample analysis of workflow evaluation quality.

### Inspect workflow results

Prints each sample’s final response, trace count, and metric scores.


In [ ]:
# Per-sample: agent_traces (one per DAG node) and workflow_execution dict
for row in results.sample_results:
    print("Final response:", row.get("final_response"))
    print("Agent traces:", len(row.get("agent_traces", [])), "nodes")
    for k, v in row.get("metrics", {}).items():
        print(f"  {k}: score={v.get('score')}")

## Summary

This notebook demonstrated how to evaluate multi-agent workflows arranged as a DAG on the FloTorch gateway.

The key components included:

1. **Gateway and DAG**: FloTorch credentials and LLM config were set; a DAG config with START, AGENT, and END nodes referenced agents deployed in the [FloTorch Console](https://docs.flotorch.cloud/introduction/).
2. **Workflow runner**: A `WorkflowRunner` was created from the DAG and used to run each test case through the graph.
3. **Dataset and traces**: A partial agent dataset was loaded from JSON; `run_on_dataset` produced full samples with traces for scoring.
4. **Evaluation**: `AgentEvaluation` ran `goal_achievement`, `response_coherence`, and `ragas:agent_goal_accuracy`; summary and per-sample results were inspected.

This example showcases evaluating multi-agent DAG workflows with Floeval.